# GEPA Skill Optimization with Evals

Optimize Claude Agent Skills using **GEPA's `optimize_anything`** API with **evaluation test cases and LLM-as-judge assertions**.

## How it differs from the static-only notebook

| | Static-only | Eval-based (this notebook) |
|---|---|---|
| **Metric** | 100% static analysis | 40% static + 60% LLM-as-judge |
| **Scoring** | Filler removal, conciseness, code blocks, structure | Same static checks + assertion pass rate |
| **Evals** | None | Test cases with verifiable assertions |
| **Validation** | Metric score only | Per-assertion PASS/FAIL verdicts |

## Pipeline

1. **Setup** -- Import dependencies, configure API key
2. **Load Skill** -- Parse skill directory
3. **Analyze** -- Check against best practices
4. **Load / Generate Evals** -- Load test cases, generate assertions
5. **Define Evaluator** -- Hybrid: static analysis + LLM-as-judge
6. **Configure** -- Objective, background, GEPA config
7. **Run GEPA** -- `optimize_anything` evolves the skill content
8. **Validate** -- Final assertion pass rates
9. **Save** -- Output optimized skill + benchmark

## 1. Setup

In [ ]:
import json
import re
import os
import sys
from pathlib import Path

import gepa.optimize_anything as oa
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig
from openai import OpenAI
from dotenv import load_dotenv

# Add repo root to path so we can import skillopt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from skillopt import SkillParser, SkillAnalyzer

load_dotenv()

parser = SkillParser()
analyzer = SkillAnalyzer()

# Verify API key and set up model
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

MODEL = "openai/gpt-4o"

def strip_provider_prefix(model: str) -> str:
    return model.removeprefix("openai/")

def _get_openai_client() -> OpenAI:
    api_base = os.getenv("OPENAI_API_BASE")
    return OpenAI(base_url=api_base) if api_base else OpenAI()

def _chat(client, model, prompt, temperature=0.7):
    response = client.chat.completions.create(
        model=strip_provider_prefix(model),
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content

def _parse_json_or_lines(text):
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict) and "assertions" in parsed:
            return parsed["assertions"]
        return []
    except json.JSONDecodeError:
        pass
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return [l.strip().lstrip("-*").strip() for l in text.strip().split("\n") if l.strip() and len(l.strip()) > 10]

print(f"Using model: {MODEL}")
print("API key: set")

## 2. Load Skill

Set `SKILL_PATH` to your skill directory and `EVALS_PATH` to your evals JSON (or `None` to auto-generate).

In [3]:
# ---- CONFIGURE THESE ----
SKILL_PATH = ROOT / "examples/example_2/dad-joke"
EVALS_PATH = None  # set to None to auto-generate
OUTPUT_DIR = None  # set a Path to save results, or leave None to skip saving
# --------------------------

skill = parser.parse_directory(SKILL_PATH)
skill_content = skill.main_file.raw_content

print(f"Loaded skill: {skill.name}")
print(f"  Lines: {skill.total_lines}")
print(f"  References: {len(skill.references)}")

Loaded skill: dad-joke
  Lines: 141
  References: 0


## 3. Analyze Skill

In [4]:
report = analyzer.analyze(skill)

errors = [i for i in report.issues if i.severity == "error"]
warnings = [i for i in report.issues if i.severity == "warning"]
suggestions = [i for i in report.issues if i.severity == "suggestion"]

print(f"Score: {report.score}/100")
print(f"Issues: {len(report.issues)} ({len(errors)} errors, {len(warnings)} warnings, {len(suggestions)} suggestions)")
print()
for issue in report.issues:
    icon = {"error": "[ERR]", "warning": "[WRN]", "suggestion": "[SUG]"}.get(issue.severity, "[?]")
    print(f"  {icon} {issue.category}: {issue.message}")

Score: 100/100
Issues: 2 (0 errors, 1 warnings, 1 suggestions)

  [SUG] workflow: Multi-step workflow without checklist
  [WRN] frontmatter: Description may not explain when to use the skill


## 4. Load / Generate Evals & Assertions

If `EVALS_PATH` points to a file, it loads the test cases from it. Otherwise, test cases are auto-generated from the skill content.

Either way, any test case missing `expectations` gets assertions auto-generated via an LLM call (3–6 per case, filtered for objectivity).

In [ ]:
# Load or generate eval cases
client = _get_openai_client()

if EVALS_PATH and Path(EVALS_PATH).exists():
    with open(EVALS_PATH) as f:
        evals_data = json.load(f)
    print(f"Loaded evals from: {EVALS_PATH}")
else:
    print("No evals file found -- auto-generating from skill content...")
    prompt = (
        "Generate evaluation test cases for a Claude Agent Skill.\n\n"
        "Create 3 realistic test prompts a user would actually say. Cover "
        "different aspects of the skill. Make prompts specific and detailed.\n\n"
        f"Skill name: {skill.name}\n"
        f"Skill content:\n{skill_content[:6000]}\n\n"
        "Return JSON: "
        '{"evals": [{"id": 1, "name": "short-name", "prompt": "realistic user prompt", '
        '"expected_output": "description of success"}]}'
    )
    response_text = _chat(client, MODEL, prompt)
    try:
        evals_data = json.loads(response_text)
        if "evals" not in evals_data:
            evals_data = {"evals": evals_data if isinstance(evals_data, list) else []}
    except json.JSONDecodeError:
        evals_data = {
            "evals": [{"id": 1, "name": "basic-usage",
                       "prompt": f"Help me with a typical {skill.name} task",
                       "expected_output": "Should provide relevant guidance", "expectations": []}]
        }
    evals_data["skill_name"] = skill.name
    print(f"  Generated {len(evals_data.get('evals', []))} eval case(s)")

eval_cases = evals_data.get("evals", [])
print(f"  Total eval cases: {len(eval_cases)}")
for ec in eval_cases:
    print(f"    [{ec.get('id')}] {ec.get('name', 'unnamed')}")

In [ ]:
# Generate assertions for any eval case that doesn't already have them

for eval_case in eval_cases:
    if eval_case.get("expectations"):
        continue

    print(f"Generating assertions for eval {eval_case.get('id')}: {eval_case.get('name')}...")
    prompt = (
        "Generate verifiable assertions for a skill evaluation test case.\n\n"
        "Good assertions: objective, specific (e.g. 'The response includes the kubectl logs command')\n"
        "Bad assertions: subjective (e.g. 'The response is helpful')\n\n"
        f"Skill content:\n{skill_content[:6000]}\n\n"
        f"User task prompt: {eval_case['prompt']}\n"
        f"Expected output: {eval_case.get('expected_output', '')}\n\n"
        'Return a JSON array of 3-6 verifiable assertion strings. Example: ["assertion 1", "assertion 2"]'
    )
    response_text = _chat(client, MODEL, prompt)
    eval_case["expectations"] = _parse_json_or_lines(response_text)

# Summary
total_assertions = sum(len(ec.get("expectations", [])) for ec in eval_cases)
print(f"\nEval cases with assertions ({total_assertions} total):")
for ec in eval_cases:
    exps = ec.get("expectations", [])
    print(f"  [{ec.get('id')}] {ec.get('name', 'unnamed')}: {len(exps)} assertion(s)")
    for a in exps:
        print(f"      - {a}")

## 5. Define Evaluator

The evaluator combines static analysis (40%) with LLM-as-judge assertion scoring (60%). It returns `(score, feedback_dict)` so GEPA's reflection LM knows exactly what to improve.

In [ ]:
def count_code_blocks(content: str) -> int:
    return len(re.findall(r'```', content)) // 2


def static_analysis_score(optimized: str, original: str) -> float:
    """Static analysis score (0.0-1.0): 30% filler, 25% conciseness, 25% code blocks, 20% structure."""
    score = 0.0
    filler_phrases = [
        'make sure to', 'ensure that', "don't forget to", 'remember to',
        'it is important to', 'please note that', 'keep in mind',
        'you should', 'you need to', 'you must',
    ]
    filler_count = sum(1 for p in filler_phrases if p.lower() in optimized.lower())
    score += 0.30 * max(0, 1 - (filler_count / 5))

    orig_len = len(original) if original else 1
    opt_len = len(optimized)
    if opt_len < orig_len:
        ratio = 1 - (opt_len / orig_len)
        if ratio < 0.3:
            score += 0.25 * (ratio / 0.3)
        elif ratio <= 0.7:
            score += 0.25
        else:
            score += 0.25 * max(0, 1 - (ratio - 0.7) * 3)

    orig_blocks = count_code_blocks(original)
    opt_blocks = count_code_blocks(optimized)
    if orig_blocks > 0:
        block_ratio = opt_blocks / orig_blocks
        score += 0.25 * (min(1.0, block_ratio) if block_ratio >= 0.5 else block_ratio * 0.5)
    else:
        score += 0.25

    has_fm = optimized.strip().startswith('---')
    has_sec = '## ' in optimized
    has_code = '```' in optimized
    score += 0.20 * ((0.4 if has_fm else 0) + (0.3 if has_sec else 0) + (0.3 if has_code else 0))
    return score


def judge_skill_against_assertions(optimized_skill, eval_cases_list, model=MODEL):
    """LLM-as-judge: evaluate skill against all assertions. Returns (score, details)."""
    judge_client = _get_openai_client()
    total_assertions = 0
    total_passed = 0
    all_results = []

    for ec in eval_cases_list:
        assertions = ec.get("expectations", [])
        if not assertions:
            continue

        prompt = (
            "Judge whether an optimized skill would guide an agent to satisfy assertions.\n\n"
            "For each assertion: PASS if the skill clearly provides the knowledge needed, "
            "FAIL if it lacks the information. Be strict.\n\n"
            f"Optimized skill:\n{optimized_skill}\n\n"
            f"User task prompt: {ec['prompt']}\n\n"
            f"Assertions:\n{json.dumps(assertions)}\n\n"
            'Return JSON array: [{"assertion": "...", "passed": true/false, "reason": "brief evidence"}]'
        )
        response_text = _chat(judge_client, model, prompt, temperature=0.0)

        try:
            verdicts = _parse_json_or_lines(response_text)
            if isinstance(verdicts, list) and verdicts and isinstance(verdicts[0], dict):
                passed = sum(1 for v in verdicts if v.get("passed", False))
                total = len(verdicts)
            else:
                passed, total, verdicts = 0, len(assertions), []
        except (TypeError, IndexError):
            passed = max(0, response_text.upper().count("PASS") - response_text.upper().count("NOT PASS"))
            total, verdicts = len(assertions), []

        total_assertions += total
        total_passed += max(0, passed)
        all_results.append({
            "eval_id": ec.get("id", 0), "eval_name": ec.get("name", ""),
            "passed": max(0, passed), "total": total,
            "pass_rate": max(0, passed) / total if total > 0 else 0,
            "verdicts": verdicts,
        })

    return (total_passed / total_assertions if total_assertions > 0 else 0), all_results


def create_eval_evaluator(original_content, eval_cases_list, model=MODEL):
    """Create evaluator closure for optimize_anything: 40% static + 60% assertions."""

    def evaluate(candidate: str) -> tuple[float, dict]:
        feedback = {}
        static = static_analysis_score(candidate, original_content)
        feedback["static_score"] = f"{static:.3f}"

        assertion_score, detailed = judge_skill_against_assertions(candidate, eval_cases_list, model)
        feedback["assertion_pass_rate"] = f"{assertion_score:.3f}"

        combined = 0.40 * static + 0.60 * assertion_score
        feedback["combined_score"] = f"{combined:.3f}"

        for r in detailed:
            status = "PASS" if r["pass_rate"] == 1.0 else f"PARTIAL ({r['passed']}/{r['total']})"
            feedback[f"eval_{r['eval_id']}_{r['eval_name']}"] = status

        oa.log(f"Static: {static:.3f} | Assertions: {assertion_score:.3f} | Combined: {combined:.3f}")
        for r in detailed:
            oa.log(f"  Eval [{r['eval_id']}] {r['eval_name']}: {r['passed']}/{r['total']}")

        return combined, feedback

    return evaluate

print("Evaluator defined: 40% static + 60% LLM-as-judge")

## 6. Configure Optimization

Set up the objective, background context, and evaluator for `optimize_anything`.

In [ ]:
# Format eval cases for background context
def format_eval_cases(cases):
    lines = []
    for ec in cases:
        lines.append(f"## Test Case {ec.get('id', '?')}: {ec.get('name', 'unnamed')}")
        lines.append(f"Prompt: {ec['prompt']}")
        lines.append(f"Expected: {ec.get('expected_output', 'N/A')}")
        for a in ec.get("expectations", []):
            lines.append(f"  - {a}")
        lines.append("")
    return "\n".join(lines)

eval_cases_str = format_eval_cases(eval_cases)

issues_str = "\n".join([
    f"- [{issue.severity}] {issue.category}: {issue.message}"
    for issue in report.issues[:20]
])

objective = (
    "Optimize this Claude Agent Skill (SKILL.md) to be effective, concise, and "
    "satisfy all evaluation criteria. The skill must contain the knowledge, commands, "
    "and workflows needed for an agent to handle all test cases and pass all assertions. "
    "The output must be a complete, valid SKILL.md file starting with --- frontmatter."
)

background = (
    f"Issues found by the skill analyzer:\n{issues_str}\n\n"
    f"Evaluation test cases the optimized skill must support:\n{eval_cases_str}\n\n"
    "Optimization guidelines:\n"
    "- Remove filler phrases: 'make sure to', 'ensure that', 'don't forget to'\n"
    "- Don't explain concepts Claude already knows (YAML, JSON, APIs, etc.)\n"
    "- Preserve frontmatter, section headers, and essential code blocks\n"
    "- Consolidate similar commands using placeholders\n"
    "- Target 30-70% size reduction"
)

evaluator = create_eval_evaluator(skill_content, eval_cases, model=MODEL)

print(f"Eval cases: {len(eval_cases)}")
print(f"Total assertions: {total_assertions}")
print("Objective, background, and evaluator ready")

## 7. Run GEPA Optimization

`optimize_anything` takes the skill content as the seed candidate and evolves it via reflection-driven search. Each candidate is scored by the hybrid evaluator. GEPA's reflection LM reads the structured feedback and proposes improvements.

In [ ]:
MAX_METRIC_CALLS = 10

print(f"Running GEPA optimize_anything (max_metric_calls={MAX_METRIC_CALLS})...")
print(f"  Metric: 40% static + 60% assertion ({total_assertions} assertions)")
print()

result = optimize_anything(
    seed_candidate=skill_content,
    evaluator=evaluator,
    objective=objective,
    background=background,
    config=GEPAConfig(
        engine=EngineConfig(
            max_metric_calls=MAX_METRIC_CALLS,
            display_progress_bar=True,
        ),
        reflection=ReflectionConfig(
            reflection_lm=MODEL,
            reflection_lm_kwargs={"temperature": 0.7},
            skip_perfect_score=True,
        ),
    ),
)

print(f"\nGEPA optimization complete")
print(f"  Best score: {result.val_aggregate_scores[result.best_idx]:.3f}")
print(f"  Candidates explored: {len(result.candidates)}")
print(f"  Total metric calls: {result.total_metric_calls}")

## 8. Validate

Extract the best candidate and run final assertion validation.

In [ ]:
# Extract optimized content
optimized_content = result.best_candidate
if isinstance(optimized_content, dict):
    optimized_content = list(optimized_content.values())[0]

# Metrics
original_len = len(skill_content)
optimized_len = len(optimized_content)
reduction_pct = ((original_len - optimized_len) / original_len * 100) if original_len > 0 else 0
orig_blocks = count_code_blocks(skill_content)
opt_blocks = count_code_blocks(optimized_content)

print(f"Original:  {original_len:,} chars, {orig_blocks} code blocks")
print(f"Optimized: {optimized_len:,} chars, {opt_blocks} code blocks")
print(f"Reduction: {original_len - optimized_len:,} chars ({reduction_pct:.1f}%)")

In [ ]:
# Final assertion validation
final_assertion_score, detailed_results = judge_skill_against_assertions(
    optimized_content, eval_cases, model=MODEL
)

static = static_analysis_score(optimized_content, skill_content)
combined = 0.40 * static + 0.60 * final_assertion_score

print(f"Scores:")
print(f"  Static analysis:     {static:.3f}")
print(f"  Assertion pass rate: {final_assertion_score:.3f}")
print(f"  Combined (40/60):    {combined:.3f}")
print()
print("Assertion results by eval:")
for r in detailed_results:
    status = "PASS" if r['pass_rate'] == 1.0 else "PARTIAL" if r['pass_rate'] > 0 else "FAIL"
    print(f"  [{r['eval_id']}] {r['eval_name']}: {r['passed']}/{r['total']} ({r['pass_rate']:.0%}) {status}")
    if isinstance(r.get('verdicts'), list):
        for v in r['verdicts']:
            icon = "✓" if v.get('passed') else "✗"
            print(f"      {icon} {v.get('assertion', '')}")

In [15]:
# Preview the optimized skill
print("Optimized SKILL.md preview:")
print("=" * 60)
print(optimized_content[:3000])
if len(optimized_content) > 3000:
    print("\n... (truncated)")

Optimized SKILL.md preview:
---
name: dad-joke
description: Generate short, pun-driven, wholesome dad jokes.
---

# Dad Joke Generator

As a dad joke expert, your goal is to produce jokes in the dad joke tradition: short, pun-driven, wholesome, and aimed at eliciting groans.

## When to use

Use this skill when the user requests a dad joke or a similar type of humor, like corny or clean jokes.

## Characteristics of Dad Jokes

- **Mandatory Puns:** Every dad joke must include a pun, relying on wordplay or double meanings.
- **Simplicity and Obviousness:** The pun should be immediately obvious, leading to a groan.
- **Wholesome:** Always appropriate for all audiences, without any offensive content.
- **Concise:** Limited to one or two sentences.
- **Enthusiastic Delivery:** Present with genuine humor as though it's the funniest joke ever.

## Joke Formats

Use and vary these formats to keep jokes engaging:

- **Question-and-Answer:** A setup question with a pun in the answer.
- **Setup-

## 9. Save Results

Set `OUTPUT_DIR` at the top of the notebook to save the optimized skill and benchmark.

In [ ]:
if OUTPUT_DIR:
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    (output_path / "SKILL.md").write_text(optimized_content)

    benchmark = {
        "metadata": {
            "skill_name": skill.name,
            "model": MODEL,
            "optimizer": "GEPA optimize_anything",
            "max_metric_calls": MAX_METRIC_CALLS,
            "metric_weights": {"static": 0.40, "assertions": 0.60},
        },
        "scores": {
            "static_analysis": round(static, 4),
            "assertion_pass_rate": round(final_assertion_score, 4),
            "combined": round(combined, 4),
        },
        "metrics": {
            "original_chars": original_len,
            "optimized_chars": optimized_len,
            "reduction_percent": round(reduction_pct, 1),
            "original_code_blocks": orig_blocks,
            "optimized_code_blocks": opt_blocks,
        },
        "assertion_results": detailed_results,
        "evals": eval_cases,
    }

    with open(output_path / "benchmark.json", "w") as f:
        json.dump(benchmark, f, indent=2)

    print(f"Saved to: {output_path}")
    print(f"  SKILL.md       — optimized skill")
    print(f"  benchmark.json — scores + assertion verdicts")
else:
    print("OUTPUT_DIR not set — skipping save. Set it in cell 2 to enable.")